# Fake Review Archaeology - Data Exploration

This notebook explores the fake review dataset and provides insights into:
- Data distribution
- Class balance
- Text characteristics
- Temporal patterns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append('../src')

from data_pipeline import DataPipeline

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

## 1. Load Sample Data

In [ ]:
# Create sample data for demonstration
np.random.seed(42)
n_samples = 5000

sample_data = {
    'review_id': range(n_samples),
    'reviewer_id': [f'user_{np.random.randint(1, 500)}' for _ in range(n_samples)],
    'product_id': [f'prod_{np.random.randint(1, 200)}' for _ in range(n_samples)],
    'category': np.random.choice(
        ['Electronics', 'Supplements', 'Clothing', 'Home & Garden', 'Books', 'Beauty'],
        n_samples
    ),
    'rating': np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.08, 0.12, 0.2, 0.3, 0.3]),
    'review_date': pd.date_range('2024-01-01', periods=n_samples, freq='H'),
    'review_text': [
        'This product exceeded my expectations! Highly recommend.',
        'Average quality, nothing special but decent.',
        'Terrible experience, would not recommend.',
        'Amazing value for money, fast shipping!',
        'Disappointed with the quality.',
    ] * (n_samples // 5) + ['Good product.'] * (n_samples % 5),
    'is_recommended': np.random.choice([0, 1], n_samples, p=[0.25, 0.75])
}

df = pd.DataFrame(sample_data)
df['review_length'] = df['review_text'].str.len()
df['word_count'] = df['review_text'].str.split().str.len()

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label distribution
label_counts = df['is_recommended'].value_counts()
axes[0].pie(label_counts.values, labels=['Genuine', 'Fake'], autopct='%1.1f%%', 
           colors=['#2ecc71', '#e74c3c'])
axes[0].set_title('Label Distribution')

# Rating distribution by label
rating_by_label = df.groupby(['rating', 'is_recommended']).size().unstack()
rating_by_label.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Rating Distribution by Label')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].legend(['Fake', 'Genuine'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3. Text Characteristics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Review length distribution
df[df['is_recommended'] == 1]['review_length'].hist(bins=50, alpha=0.7, 
                                                      label='Genuine', ax=axes[0,0])
df[df['is_recommended'] == 0]['review_length'].hist(bins=50, alpha=0.7, 
                                                      label='Fake', ax=axes[0,0])
axes[0,0].set_title('Review Length Distribution')
axes[0,0].set_xlabel('Characters')
axes[0,0].legend()

# Word count distribution
df[df['is_recommended'] == 1]['word_count'].hist(bins=50, alpha=0.7, 
                                                   label='Genuine', ax=axes[0,1])
df[df['is_recommended'] == 0]['word_count'].hist(bins=50, alpha=0.7, 
                                                   label='Fake', ax=axes[0,1])
axes[0,1].set_title('Word Count Distribution')
axes[0,1].set_xlabel('Words')
axes[0,1].legend()

# Box plot of review length by category
df.boxplot(column='review_length', by='category', ax=axes[1,0])
axes[1,0].set_title('Review Length by Category')
axes[1,0].set_xlabel('Category')
plt.setp(axes[1,0].xaxis.get_majorticklabels(), rotation=45)

# Average metrics by category
cat_stats = df.groupby('category').agg({
    'review_length': 'mean',
    'word_count': 'mean',
    'is_recommended': 'mean'
})
cat_stats.plot(kind='bar', ax=axes[1,1])
axes[1,1].set_title('Average Metrics by Category')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].legend(['Avg Length', 'Avg Words', 'Genuine Rate'])

plt.tight_layout()
plt.show()

## 4. Temporal Patterns

In [ ]:
df['date'] = pd.to_datetime(df['review_date']).dt.date
df['hour'] = pd.to_datetime(df['review_date']).dt.hour
df['dayofweek'] = pd.to_datetime(df['review_date']).dt.dayofweek

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reviews over time
daily_counts = df.groupby('date').size()
daily_counts.plot(ax=axes[0,0])
axes[0,0].set_title('Reviews Over Time')
axes[0,0].set_xlabel('Date')

# Hour of day distribution
hourly = df.groupby('hour').agg({'is_recommended': ['count', 'mean']})
hourly.columns = ['count', 'genuine_rate']
hourly['count'].plot(ax=axes[0,1])
axes[0,1].set_title('Reviews by Hour of Day')
axes[0,1].set_xlabel('Hour')

# Day of week distribution
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow = df.groupby('dayofweek').agg({'is_recommended': ['count', 'mean']})
dow.columns = ['count', 'genuine_rate']
dow.index = dow_names
dow['count'].plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Reviews by Day of Week')
axes[1,0].tick_params(axis='x', rotation=0)

# Fraud rate by hour
hourly['genuine_rate'].plot(ax=axes[1,1])
axes[1,1].set_title('Genuine Rate by Hour')
axes[1,1].set_xlabel('Hour')
axes[1,1].axhline(y=0.5, color='r', linestyle='--')

plt.tight_layout()
plt.show()

## 5. Category Analysis

In [ ]:
# Category statistics
cat_analysis = df.groupby('category').agg({
    'review_id': 'count',
    'is_recommended': 'mean',
    'review_length': 'mean',
    'rating': 'mean'
}).round(2)
cat_analysis.columns = ['Total Reviews', 'Genuine Rate', 'Avg Length', 'Avg Rating']

print("Category Analysis:")
print(cat_analysis.sort_values('Genuine Rate'))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by category
fraud_rate = 1 - cat_analysis['Genuine Rate']
fraud_rate.sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='#e74c3c')
axes[0].set_title('Fraud Rate by Category')
axes[0].set_xlabel('Fraud Rate')

# Review count by category
cat_analysis['Total Reviews'].plot(kind='bar', ax=axes[1], color='#3498db')
axes[1].set_title('Review Count by Category')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix
numeric_cols = ['rating', 'review_length', 'word_count', 'is_recommended', 'hour']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

print("\nKey Correlations with 'is_recommended':")
print(corr_matrix['is_recommended'].sort_values(ascending=False))

## 7. Summary Statistics

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\nTotal Reviews: {len(df):,}")
print(f"Unique Reviewers: {df['reviewer_id'].nunique():,}")
print(f"Unique Products: {df['product_id'].nunique():,}")
print(f"Categories: {df['category'].nunique()}")

print(f"\nLabel Distribution:")
print(f"  Genuine: {(df['is_recommended'] == 1).sum():,} ({(df['is_recommended'] == 1).mean():.1%})")
print(f"  Fake: {(df['is_recommended'] == 0).sum():,} ({(df['is_recommended'] == 0).mean():.1%})")

print(f"\nReview Length Statistics:")
print(df['review_length'].describe())

print(f"\nRating Distribution:")
print(df['rating'].value_counts().sort_index())

## Next Steps

1. Proceed to `02_feature_analysis.ipynb` for feature engineering exploration
2. Run the training pipeline with `python train.py`
3. Explore the Streamlit dashboard with `streamlit run dashboard/app.py`